In [21]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

In [22]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories
from pathlib import Path
import os

# Ensure PROJECT_ROOT is correctly set for notebook context
# Look for config folder to identify project root
current_dir = Path(os.getcwd())
while current_dir != current_dir.parent:
    if (current_dir / "config" / "config.yaml").exists():
        PROJECT_ROOT = current_dir
        break
    current_dir = current_dir.parent
else:
    # Fallback: assume we're one level deeper than expected
    PROJECT_ROOT = Path(os.getcwd()).parent if not (Path(os.getcwd()) / "config").exists() else Path(os.getcwd())

In [23]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        # Convert relative paths to absolute paths
        root_dir = Path(config.root_dir) if Path(
            config.root_dir).is_absolute() else PROJECT_ROOT / config.root_dir
        data_path = Path(config.data_path) if Path(
            config.data_path).is_absolute() else PROJECT_ROOT / config.data_path

        create_directories([root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=root_dir,
            data_path=data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config

In [24]:
import os
from textsummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [25]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    
    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)
        
        # For Pegasus tokenizer, we don't need as_target_tokenizer()
        target_encodings = self.tokenizer(example_batch['summary'], max_length=128, truncation=True)
            
        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }
    

    def convert(self):
        dataset_samsum = load_from_disk(str(self.config.data_path))
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
        dataset_samsum_pt.save_to_disk(os.path.join(str(self.config.root_dir), "samsum_dataset"))

In [26]:
try:
    # First verify the dataset actually exists
    import os
    artifacts_path = Path(os.getcwd()) / "artifacts" / "data_ingestion"
    print(f"Current working directory: {os.getcwd()}")
    print(f"Artifacts path: {artifacts_path}")
    print(f"Artifacts path exists: {artifacts_path.exists()}")
    if artifacts_path.exists():
        print(f"Contents of artifacts/data_ingestion: {os.listdir(artifacts_path)}")
    
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    
    # Debug: Print resolved paths
    print(f"\nData path: {data_transformation_config.data_path}")
    print(f"Data path type: {type(data_transformation_config.data_path)}")
    print(f"Root dir: {data_transformation_config.root_dir}")
    print(f"Root dir type: {type(data_transformation_config.root_dir)}")
    print(f"Data path exists: {Path(data_transformation_config.data_path).exists()}")
    
    if not Path(data_transformation_config.data_path).exists():
        print(f"\nERROR: Dataset not found at {data_transformation_config.data_path}")
        print("Please ensure data ingestion stage has completed successfully")
    else:
        data_transformation = DataTransformation(config=data_transformation_config)
        data_transformation.convert()
        print("\nData transformation completed successfully!")
        
except Exception as e:
    import traceback
    traceback.print_exc()
    raise e

Current working directory: c:\Users\jayan\Desktop\text-summarization-project\research
Artifacts path: c:\Users\jayan\Desktop\text-summarization-project\research\artifacts\data_ingestion
Artifacts path exists: False
[2026-04-06 13:48:27,404: INFO: common: yaml file: C:\Users\jayan\Desktop\text-summarization-project\config\config.yaml loaded successfully]
[2026-04-06 13:48:27,407: INFO: common: yaml file: C:\Users\jayan\Desktop\text-summarization-project\params.yaml loaded successfully]
[2026-04-06 13:48:27,409: INFO: common: created directory at: artifacts]
[2026-04-06 13:48:27,412: INFO: common: created directory at: c:\Users\jayan\Desktop\text-summarization-project\artifacts\data_transformation]

Data path: c:\Users\jayan\Desktop\text-summarization-project\artifacts\data_ingestion\samsum_dataset
Data path type: <class 'pathlib.WindowsPath'>
Root dir: c:\Users\jayan\Desktop\text-summarization-project\artifacts\data_transformation
Root dir type: <class 'pathlib.WindowsPath'>
Data path e

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 110682.65 examples/s]


Data transformation completed successfully!
